# Tokenization and N-gram Language Models

Text processing from first principles: regular-expression tokenizers, edit
distance, subword tokenizers trained on Persian, and n-gram language models with
four smoothing strategies compared by perplexity.

Everything here is implemented directly rather than called from a library, so
the mechanics stay visible — the n-gram model is a `defaultdict` of counts, and
the smoothing methods differ only in how they turn those counts into a
probability.

**Headline result:** unsmoothed 4-gram perplexity is infinite, Laplace gives
3,494, and backoff gives **35.9** — a 97× improvement from changing nothing but
how unseen contexts are handled.

## 1. Extracting and validating emails with regex

Two patterns doing different jobs: one *extracts* `name=..., email=...` pairs
from free text, the other *validates* each address. Splitting them matters —
a permissive extraction pattern followed by a strict validation pattern reports
which addresses are malformed, whereas one combined pattern would silently skip
them.

In [ ]:
import re


extraction_pattern = r"name=([^,]+), email=([^\s,]+)"
email_validation_pattern = r"^[a-zA-Z0-9_-]+(?:\.[a-zA-Z0-9_-]+)*@(?:[a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}$"
with open('emails.txt', 'r') as file:
    content = file.read()

entries = re.findall(extraction_pattern, content)

valid = []
print(" Validation Results -")

for name, email in entries:
    if not re.fullmatch(email_validation_pattern, email):
        print(f"Name: {name.strip()}, Email: '{email}' -> Invalid (Wrong format)")
        continue
    local_part = email.split('@')[0].lower()
    name_parts = name.strip().lower().split()
    dynamic_name_pattern_parts = []
    for part in name_parts:
        dynamic_name_pattern_parts.append(f"(?=.*{re.escape(part)})")
    dynamic_name_pattern = r"^" + "".join(dynamic_name_pattern_parts)
    if re.search(dynamic_name_pattern, local_part):
        print(f"Name: {name.strip()}, Email: '{email}' -> Valid (Format and name match)")
        valid.append(email)
    else:
        print(f"Name: {name.strip()}, Email: '{email}' -> Valid (Format only, name mismatch)")
print(" Final List (Valid Format + Name Match) ")
print(valid)

### Searching the validated set

Filtering the addresses that passed validation.

In [ ]:
import re
# Requires the previous cell: it produces the list of valid emails.
def name_search(first, last, valid):
    pattern1_part = f"{first}[\._]*{last}"
    pattern2_part = f"{last}[\._]*{first}"
    pattern = rf"^({pattern1_part}|{pattern2_part})@.*"
    for email in valid:
        if re.fullmatch(pattern, email, re.I):
            return email
    return None 
for i in range(2):
 name = input("name: ")
 first, last = name.split(' ')
 result = name_search(first, last, valid)
 print(result)


## 2. Minimum edit distance

Levenshtein distance with the standard dynamic-programming table: insertion,
deletion, and substitution each cost 1, and cell `(i, j)` is the cheapest way to
turn the first `i` characters into the first `j`.

In [ ]:
def levenshtein(s1, s2):
    m, n = len(s1), len(s2)
    D = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        D[i][0] = i
    for j in range(n + 1):
        D[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 2
            D[i][j] = min(
                D[i - 1][j] + 1,      
                D[i][j - 1] + 1,      
                D[i - 1][j - 1] + cost  #فرض بر این است که جایگزینی هزینه ای برابر با 2 دارد
            )
    return D[m][n]

pair1 = "Athletic", "Atlantic"
pair2 = "London", "Boston"
pair3 = "Action", "Compact"
pair4 = "", "Sting"
print(f'pair1: {levenshtein(pair1[0], pair1[1])}')
print(f'pair2: {levenshtein(pair2[0], pair2[1])}')
print(f'pair3: {levenshtein(pair3[0], pair3[1])}')
print(f'pair4: {levenshtein(pair4[0], pair4[1])}')

### Spelling correction from edit distance

Using the distance function as an autocorrector: for each out-of-vocabulary
word, propose the vocabulary entry with the smallest distance.

Short words are the weak point. At two or three characters almost every
vocabulary word is within distance 1–2, so the "nearest" candidate is close to
arbitrary — which is why `at`, `in`, `it` and `the` are handled as a special
case instead.

In [ ]:
import string
import os
# Requires the minimum-edit-distance function defined in the previous cell.
with open('vocab.txt', 'r') as f:
     content = f.read() 
     words_list = content.split(',') 
     vocab = set(word.strip().lower() for word in words_list if word.strip())
missing_words = {'at',  'in',  'it', 'the'}#میتوانستیم فرض کنیم که کلا کلماتی که 3 حرفی یا دو حرفی هستند را چک نکند(چون کمتر اشبتاه میشوند) ، کار دیگری که میتوان کرد که این کلمات را اضافه کرد
vocab.update(missing_words)
vocab.remove('python') #age capital mohem bashe in ro fa al mikonim
vocab.add('Python')

sentence_to_be_corrected = "helo studnts at the universty are wrting ther frst edit distnce algorthm in pythn, and they reely enjy it!"
words = sentence_to_be_corrected.split()
corrected_words = []


max_length_difference = 2 

for word in words:
    punctuation = ''
    if word and word[-1] in string.punctuation:
        punctuation = word[-1]
        word_cleaned = word[:-1].lower()
    else:
        word_cleaned = word.lower()
    if not word_cleaned:
        continue
    if word_cleaned in vocab:
        corrected_words.append(word) 
        continue
    current_len = len(word_cleaned)
    

    candidates = [
        v for v in vocab 
        if abs(len(v) - current_len) <= max_length_difference#هر کلمه با کلماتی که کمتر  یا بیشتر از دو(یعنی 2 کاراکتر بیشتر یا کمتر) احتلاف دارند برای بررسی مینیم دیستنس انتخاب میشوند
    ]
    
    if not candidates:
        
        corrected_words.append(word)
        continue

    min_dist = float('inf')
    best_match = word_cleaned 

    for vocab_word in candidates:
        dist = levenshtein(word_cleaned, vocab_word)
        
        if dist < min_dist:
            min_dist = dist
            best_match = vocab_word
            
            if dist == 1:
                break 
    
    print(f"'{word_cleaned}' -> '{best_match}' (dist: {min_dist})")
    corrected_words.append(best_match + punctuation)


final_sentence = ' '.join(corrected_words)

print(f" actual:   {sentence_to_be_corrected}")
print(f"revised: {final_sentence}")

## 3. Rule-based tokenization

A first tokenizer: alternate between letter runs, digit runs, and single
characters. It works on plain prose and fails in the predictable places —
`U.S.A.` fragments, `$12.40` splits at the decimal point, and `poster-print`
becomes two tokens.

In [ ]:
import re
pattern = r"([A-Za-z]+)|(\d+)|(.)"
sentence1 = "Hello, world! NLP is fun."
sentence2 = "That U.S.A. poster-print costs $12.40..."
print(f"sen1: {sentence1}")
print(f"token: { [group for match_tuple in re.findall(pattern, sentence1, re.VERBOSE)for group in match_tuple if group] }")
print('\n')
print(f"sen2: {sentence2}")
print(f"token: {[group for match_tuple in re.findall(pattern, sentence2, re.VERBOSE) for group in match_tuple if group] }")

### A tokenizer that handles the hard cases

The second pattern adds ordered alternatives for ellipses, acronyms, currency
amounts, and hyphenated or apostrophised words. Order is what makes it work:
the acronym branch has to be tried before the single-character fallback, or
`U.S.A.` is consumed one dot at a time.

In [ ]:
pattern2 = r"(\.\.\.) |((?:[A-Za-z]\.){2,}) |  (\$?\d+(?:\.\d+)?) |([A-Za-z]+(?:(?:'|-)[A-Za-z]+)*) | (.) "     
sentence1 = "Hello, world! NLP is fun."
sentence2 = "That U.S.A. poster-print costs $12.40..."
print(f"sen1: {sentence1}")
print(f"token: { [group for match_tuple in re.findall(pattern2, sentence1, re.VERBOSE)for group in match_tuple if group] }")
print('\n')
print(f"sen2: {sentence2}")
print(f"token: {[group for match_tuple in re.findall(pattern2, sentence2, re.VERBOSE) for group in match_tuple if group] }")


## 4. Subword tokenizers on Persian

Byte-Pair Encoding trained on the TinyStories-Farsi corpus with a 30k vocabulary.

BPE starts from bytes and repeatedly merges the most frequent adjacent pair, so
it can encode any string without an `[UNK]` token — useful for Persian, where
prefixes and suffixes attach freely and a fixed word vocabulary would miss a
long tail of inflected forms.

In [ ]:
import sys
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
def get_text_iterator(split="train"):
    dataset = load_dataset("taesiri/TinyStories-Farsi", split=split, streaming=True)
    for example in dataset:
        if 'Persian' in example: 
            yield example['Persian'] 
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(
    vocab_size=30000,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)
tokenizer.train_from_iterator(get_text_iterator(), trainer=trainer)
tokenizer_path = "TinyStories-Farsi-BPE.json"
tokenizer.save(tokenizer_path)
test_sentence = "روزی یک مرد ثروتمند، پسر بچه کوچکش را بـه ده برد تا بـه او نشان دهد مردمی که در آنجا زندگی می‌کنند، چقدر فقیر هستند."
print(f"\nجمله ورودی:\n{test_sentence}")
output = tokenizer.encode(test_sentence)
print(output.tokens)
print(output.ids)
decoded_text = tokenizer.decode(output.ids)
for i in output.ids:
    decoded_text = tokenizer.decode([i])
    print(f"ID: {i: <5}  =>  Token: {decoded_text}")

### WordPiece for comparison

Same corpus, same vocabulary size. WordPiece picks the merge that most improves
the likelihood of the training data rather than the merge that is simply most
frequent, and marks continuations with `##`.

In [ ]:
import sys
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import BertPreTokenizer

tokenizer2 = Tokenizer(WordPiece(unk_token="[UNK]"))

tokenizer2.pre_tokenizer = BertPreTokenizer()


trainer2 = WordPieceTrainer(
    vocab_size=30000,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    continuing_subword_prefix="##" 
)


tokenizer2.train_from_iterator(get_text_iterator(), trainer=trainer2)
tokenizer2_path = "TinyStories-Farsi-WordPiece.json"
tokenizer2.save(tokenizer_path)
test_sentence2 = "روزی یک مرد ثروتمند، پسر بچه کوچکش را بـه ده برد تا بـه او نشان دهد مردمی که در آنجا زندگی می‌کنند، چقدر فقیر هستند."
print(f"\nجمله ورودی:\n{test_sentence}")
output2 = tokenizer2.encode(test_sentence2)
print(output2.tokens)
print(output2.ids)
decoded_text2 = tokenizer2.decode(output2.ids)
print(decoded_text2)
for i in output2.ids:
    decoded_text2 = tokenizer2.decode([i])
    print(f"ID: {i: <5}  =>  Token: {decoded_text2}")

## 5. Inspecting the corpus before preprocessing

What the raw text actually contains — Unicode normal form, digits, emoji, and
kashida elongation. This runs *before* any preprocessing is written, so the
cleaning steps in the next cell are a response to measurements rather than
guesses.

In [ ]:
import re
import unicodedata
from datasets import load_dataset
import sys
# Inspect the data first, to decide which preprocessing steps it needs.

DATASET_NAME = "taesiri/TinyStories-Farsi"
SPLIT = "train"
TEXT_COLUMN = "Persian"
SAMPLE_SIZE = 1000  

# Count digits, emoji and similar, and check for kashida-elongated words.
KASHIDA_PATTERN = re.compile(r'\u0640')
WHITESPACE_PATTERN = re.compile(r'\s{2,}|\n|\t') 
NUMBER_PATTERN = re.compile(r'\d')
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  
    "\U0001F300-\U0001F5FF"  
    "\U0001F680-\U0001F6FF" 
    "\U0001F1E0-\U0001F1FF"  
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

def check_unicode_form(text):

    return unicodedata.normalize('NFC', text) != text

def explore_dataset(dataset, text_column, sample_size):
   
    print(f"Starting exploration of {sample_size} examples...")
    

    counters = {
        "total_checked": 0,
        "kashida_found": 0,
        "emoji_found": 0,
        "extra_whitespace_found": 0,
        "numbers_found": 0,
        "non_nfc_unicode_found": 0,
        "empty_strings": 0
    }
    
   
  
    for example in dataset.take(sample_size):
        counters["total_checked"] += 1
        
        if text_column not in example or example[text_column] is None:
            counters["empty_strings"] += 1
            continue
            
        text = example[text_column]
        

        if KASHIDA_PATTERN.search(text):
            counters["kashida_found"] += 1
            
        if EMOJI_PATTERN.search(text):
            counters["emoji_found"] += 1
            
        if WHITESPACE_PATTERN.search(text):
            counters["extra_whitespace_found"] += 1
            
        if NUMBER_PATTERN.search(text):
            counters["numbers_found"] += 1
            
        if check_unicode_form(text):
            counters["non_nfc_unicode_found"] += 1
            
       
            
  
        
    return counters

def print_report(counters):
   
    if counters is None:
        return
        
    
    print("  DATA EXPLORATION REPORT")
    print(f"Total examples checked: {counters['total_checked']}")
    print(f"Empty or missing text:  {counters['empty_strings']}\n")
    
    print("Problem Analysis ")
    print(f"Examples with Kashida:      {counters['kashida_found']}")
    print(f"Examples with Emojis:       {counters['emoji_found']}")
    print(f"Examples with Extra Whitespace: {counters['extra_whitespace_found']}")
    print(f"Examples with Numbers:      {counters['numbers_found']}")
    print(f"Examples with non-NFC Unicode: {counters['non_nfc_unicode_found']}")


def main():
    print(f"Loading dataset '{DATASET_NAME}' (streaming)...")

    dataset = load_dataset(DATASET_NAME, split=SPLIT, streaming=True)
   
        
    counters = explore_dataset(dataset, TEXT_COLUMN, SAMPLE_SIZE)
    print_report(counters)

if __name__ == "__main__":
    main()


### The preprocessing pipeline

Three steps, each justified by the inspection above:

1. **NFKC normalization** — collapses Arabic/Persian codepoint variants (the
   Arabic ي and Persian ی are distinct codepoints that render near-identically)
   so they stop being separate tokens.
2. **Kashida removal** — `U+0640` is a purely decorative stretching character
   and carries no meaning.
3. **Whitespace collapsing** — repeated spaces, tabs, and newlines to a single
   space.

In [ ]:
import re
import os
import sys
import unicodedata
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder 
def normalize_unicode(text):
 
    if text is None: return 
    return unicodedata.normalize('NFC', text)

def remove_kashida(text):

    if text is None: return 
    return text.replace('\u0640', '') 

def normalize_whitespace(text):
    
    if text is None: return ""
    text = re.sub(r'\s+', ' ', text) 
    return text.strip()

def preprocess_pipeline(text):
    # These steps follow from what the dataset inspection above found.
    #text = normalize_unicode(text)
    #text = remove_kashida(text)
    #text = remove_emojis(text)
    text = normalize_whitespace(text) 
    return text


dataset = load_dataset("taesiri/TinyStories-Farsi", split="train", streaming=True)

def apply_preprocessing(example):
    example['Persian_cleaned'] = preprocess_pipeline(example['Persian'])
    return example


processed_dataset = dataset.map(apply_preprocessing)
tokenizer_path = "TinyStories-Farsi-BPE.json"
tokenizer = Tokenizer.from_file(tokenizer_path)
tokenizer.decoder = ByteLevelDecoder()

example = next(iter(processed_dataset))

original_text = example['Persian']
cleaned_text = example['Persian_cleaned']

print(f"جمله اصلی (خام):\n{original_text}\n")
print(f"جمله پیش‌پردازش شده:\n{cleaned_text}\n")

output = tokenizer.encode(cleaned_text)

print("-" * 20)
print("توکن‌های BPE :")

readable_tokens = [tokenizer.decode([i]) for i in output.ids]
print(readable_tokens)

print("\nID های توکن‌ها:")
print(output.ids)



## 6. N-gram language model

Counting n-grams and generating text. `train` builds a nested
`context -> {token: count}` map; `generate` samples from that distribution.

In [ ]:
import re
import os
import sys
import unicodedata
import random
import math
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from collections import defaultdict


class NGramLanguageModel:
    def __init__(self, n):
        if n < 2:
             print(f"Warning: n={n} is less than 2. Setting n=2.")
             self.n = 2
        else:
            self.n = n
        self.model = defaultdict(lambda: defaultdict(int))
        self.START_TOKEN = -1 
        self.END_TOKEN = -2   

    def _get_ngrams(self, token_ids_list):
        padding = [self.START_TOKEN] * (self.n - 1)
        full_sequence = padding + token_ids_list + [self.END_TOKEN]
        for i in range(len(full_sequence) - self.n + 1):
            ngram = tuple(full_sequence[i : i + self.n])
            yield ngram

    def train(self, tokenized_corpus):
        print(f"Starting training for {self.n}-gram model...")
        for token_ids_list in tokenized_corpus:
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                self.model[context][token] += 1
        

    def generate(self, num_tokens=100):
        
        print(f"\n---  Generating text with {self.n}-gram ---")
        context = tuple([self.START_TOKEN] * (self.n - 1))
        generated_ids = []
        for _ in range(num_tokens):
            if context not in self.model:
                # Sparsity problem: stop generation
                break
            possible_next_tokens = list(self.model[context].keys())
            weights = list(self.model[context].values())
            next_token = random.choices(possible_next_tokens, weights=weights, k=1)[0]
            if next_token == self.END_TOKEN:#اگر به توکن نهایی رسیده تا 100 توکن ادامه بده
                context = tuple([self.START_TOKEN] * (self.n - 1))
            else:
                generated_ids.append(next_token)
                context = tuple(list(context[1:]) + [next_token])
        return [tid for tid in generated_ids if tid != self.END_TOKEN]
    
 
    



def normalize_whitespace(text):
    if text is None: return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_pipeline(text):
    
    text = normalize_whitespace(text)
    return text
    

TOKENIZER_PATH = "TinyStories-Farsi-BPE.json"
N_SAMPLES = 50000 


    
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = ByteLevelDecoder()
vocab_size = tokenizer.get_vocab_size() # Get vocab size for perplexity
print(f"BPE tokenizer loaded from '{TOKENIZER_PATH}'. (Vocab size: {vocab_size})")


print(f"Loading and tokenizing {N_SAMPLES} samples from the dataset...")
tokenized_corpus_ids = []


dataset = load_dataset("taesiri/TinyStories-Farsi", split="train", streaming=True)
    
for i, example in enumerate(dataset.take(N_SAMPLES)):
    cleaned_text = preprocess_pipeline(example['Persian'])
    if cleaned_text: # Ensure text is not empty
        ids = tokenizer.encode(cleaned_text).ids
        tokenized_corpus_ids.append(ids)
    
        
print(f"Processing complete. (Total sentences: {len(tokenized_corpus_ids)})")




print("\n--- Training 2-gram ---")
model_2gram = NGramLanguageModel(n=2)
model_2gram.train(tokenized_corpus_ids)

print("\n--- Training 4-gram ---")
model_4gram = NGramLanguageModel(n=4)
model_4gram.train(tokenized_corpus_ids)

print("\n- Training 8-gram -")
model_8gram = NGramLanguageModel(n=8)
model_8gram.train(tokenized_corpus_ids)



print("\n2-gram model trained.")
print("4-gram model trained.")
print("8-gram model trained.") 



ids_2 = model_2gram.generate(num_tokens=100)
text_2 = tokenizer.decode(ids_2)

ids_4 = model_4gram.generate(num_tokens=100)
text_4 = tokenizer.decode(ids_4)

ids_8 = model_8gram.generate(num_tokens=100)
text_8 = tokenizer.decode(ids_8)



print("          Final Output: Generated Texts        ")


print("\n--- 2-gram (Bigram) ---")
print(text_2)

print("\n--- 4-gram ---")
print(text_4)

print("\n--- 8-gram ---")
print(text_8)



### Perplexity, and why it comes back infinite

Perplexity is the exponentiated average negative log-likelihood — the model's
uncertainty per token, so lower is better.

Every order tested returns **infinity**. This is correct behaviour, not a bug:
maximum-likelihood estimation gives an unseen n-gram probability exactly zero,
one zero makes the product zero, and the log of zero is `-inf`. Any held-out
text contains at least one unseen context, so unsmoothed perplexity is
unbounded almost by construction. Smoothing is what fixes it.

In [ ]:
import re
import os
import sys
import unicodedata
import random
import math 
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from collections import defaultdict


class NGramLanguageModel:
 
    def __init__(self, n):
        if n < 2:
             print(f"Warning: n={n} is less than 2. Setting n=2.")
             self.n = 2
        else:
            self.n = n
        self.model = defaultdict(lambda: defaultdict(int))
        self.START_TOKEN = -1 # <S>
        self.END_TOKEN = -2    # </S>

    def _get_ngrams(self, token_ids_list):
        padding = [self.START_TOKEN] * (self.n - 1)
        full_sequence = padding + token_ids_list + [self.END_TOKEN]
        for i in range(len(full_sequence) - self.n + 1):
            ngram = tuple(full_sequence[i : i + self.n])
            yield ngram

    def train(self, tokenized_corpus):
        print(f"Starting training for {self.n}-gram model...")
        for token_ids_list in tokenized_corpus:
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                self.model[context][token] += 1
        print(f"Training for {self.n}-gram complete. Contexts found: {len(self.model)}")

    def calculate_perplexity(self, tokenized_validation_corpus, vocab_size):
      
     
        print(f"Calculating Perplexity for {self.n}-gram model ...")
        total_log_prob = 0.0
        total_token_count = 0
    
        
        has_zero_probability = False

        for token_ids_list in tokenized_validation_corpus:
            # +1 for the </S> token
            current_sentence_tokens = len(token_ids_list) + 1
            total_token_count += current_sentence_tokens
            
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                
            
                token_count_in_context = self.model[context].get(token, 0)
                
               
                context_total_count = sum(self.model[context].values())
                
                if context_total_count == 0:
               
                    has_zero_probability = True
                    break 
                
                if token_count_in_context == 0:
                    
                    has_zero_probability = True
                    break 
                    
            
                probability = token_count_in_context / context_total_count
                
                total_log_prob += math.log(probability)

            if has_zero_probability:
                break

        if has_zero_probability:
            return float('inf')
        
        avg_neg_log_likelihood = -total_log_prob / total_token_count
        perplexity = math.exp(avg_neg_log_likelihood)
        
        return perplexity


def normalize_whitespace(text):
    if text is None: return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_pipeline(text):
    text = normalize_whitespace(text)
    return text



TOKENIZER_PATH = "TinyStories-Farsi-BPE.json"
N_TRAIN_SAMPLES = 50000
N_VALID_SAMPLES = 5000



tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = ByteLevelDecoder()
vocab_size = tokenizer.get_vocab_size()
print(f"BPE tokenizer loaded. (Vocabulary size: {vocab_size})")

def load_and_tokenize(split, n_samples):
    print(f"Loading and processing {n_samples} samples from '{split}' split...")
    tokenized_ids = []
    
    dataset = load_dataset("taesiri/TinyStories-Farsi", split=split, streaming=True)
    for i, example in enumerate(dataset.take(n_samples)):
        
        cleaned_text = preprocess_pipeline(example['Persian'])
        if cleaned_text:
            ids = tokenizer.encode(cleaned_text).ids
            tokenized_ids.append(ids)
    print(f"Processing '{split}' complete. (Total sentences: {len(tokenized_ids)})")
    return tokenized_ids
    


tokenized_corpus_ids = load_and_tokenize("train", N_TRAIN_SAMPLES)
tokenized_validation_ids = load_and_tokenize("validation", N_VALID_SAMPLES)



# Train Models 
model_2gram = NGramLanguageModel(n=2)
model_2gram.train(tokenized_corpus_ids)

model_4gram = NGramLanguageModel(n=4)
model_4gram.train(tokenized_corpus_ids)

model_8gram = NGramLanguageModel(n=8)
model_8gram.train(tokenized_corpus_ids)



print("     Calculating Perplexity on Validation Data     ")

ppl_2 = model_2gram.calculate_perplexity(tokenized_validation_ids, vocab_size)
ppl_4 = model_4gram.calculate_perplexity(tokenized_validation_ids, vocab_size)
ppl_8 = model_8gram.calculate_perplexity(tokenized_validation_ids, vocab_size)


print("              Final Results                ")


print(f"\n--- 1. Validation Dataset Summary ---")
print(f"Evaluation sentences: {len(tokenized_validation_ids)}")

print("\n--- 2. Perplexity Results by N-gram ---")
print(f"Perplexity (2-gram): {ppl_2:.2f}")
print(f"Perplexity (4-gram): {ppl_4:.2f}")
print(f"Perplexity (8-gram): {ppl_8:.2f}")


### Laplace smoothing

Add 1 to every count and add the vocabulary size to every denominator, so no
n-gram has zero probability.

Perplexity becomes finite: **3,494.58** for the 4-gram on 5,000 validation
sentences. Finite but poor — add-1 takes probability mass from observed events
and spreads it over a vocabulary-sized set of mostly-impossible ones, and with
`V` in the tens of thousands that is a heavy tax.

Per-sentence perplexity varies by 4×, from 3,723 for a common phrase to 14,428
for a rare one, which is the expected signal: the model is genuinely more
surprised by unusual word sequences.

In [ ]:
import re
import os
import math
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from collections import defaultdict

class NGramLanguageModel:

    def __init__(self, n):
        if n < 2:
            print(f"Warning: n={n} is less than 2. Setting n=2.")
            self.n = 2
        else:
            self.n = n
       
        self.model = defaultdict(lambda: defaultdict(int))
        self.START_TOKEN = -1 # <S>
        self.END_TOKEN = -2   # </S>

    def _get_ngrams(self, token_ids_list):
   
        padding = [self.START_TOKEN] * (self.n - 1)
        full_sequence = padding + token_ids_list + [self.END_TOKEN]
        for i in range(len(full_sequence) - self.n + 1):
            ngram = tuple(full_sequence[i : i + self.n])
            yield ngram

    def train(self, tokenized_corpus):
    
        print(f"Starting training for {self.n}-gram model...")
        for token_ids_list in tokenized_corpus:
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                self.model[context][token] += 1
        print(f"Training for {self.n}-gram complete. Contexts found: {len(self.model)}")

    def calculate_perplexity(self, tokenized_validation_corpus, vocab_size):
        
        print(f"Calculating Perplexity for {self.n}-gram (with Laplace Smoothing)...")
        total_log_prob = 0.0
        total_token_count = 0
        
  
        V = vocab_size 

        for token_ids_list in tokenized_validation_corpus:
            # +1 for the </S> token
            current_sentence_tokens = len(token_ids_list) + 1
            total_token_count += current_sentence_tokens
            
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                
        
                
           
                token_count_in_context = self.model[context].get(token, 0)
                
             
                context_total_count = sum(self.model[context].values())
                
                # Apply Laplace (add-1) smoothing
                # numerator: (count + 1)
                numerator = token_count_in_context + 1
                
                # denominator: (total context count + V)
                denominator = context_total_count + V
              
                
              
                probability = numerator / denominator
                
                total_log_prob += math.log(probability)

        if total_token_count == 0:
            return float('inf') 

      
        avg_neg_log_likelihood = -total_log_prob / total_token_count
        
       
        perplexity = math.exp(avg_neg_log_likelihood)
        
        return perplexity



def normalize_whitespace(text):
    if text is None: return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_pipeline(text):
    text = normalize_whitespace(text)

    return text



TOKENIZER_PATH = "TinyStories-Farsi-BPE.json"
N_TRAIN_SAMPLES = 50000 
N_VALID_SAMPLES = 5000   



tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = ByteLevelDecoder()
vocab_size = tokenizer.get_vocab_size()
print(f"BPE tokenizer loaded. (Vocabulary size: {vocab_size})")

def load_and_tokenize(split, n_samples):

    print(f"\nLoading and processing {n_samples} samples from '{split}' split...")
    tokenized_ids = []
    
   
    dataset = load_dataset("taesiri/TinyStories-Farsi", split=split, streaming=True)
    
    valid_samples_found = 0
    total_items_processed = 0
    
 
    for example in dataset:
        total_items_processed += 1
        

        cleaned_text = preprocess_pipeline(example.get('Persian'))
        
        if cleaned_text:
            ids = tokenizer.encode(cleaned_text).ids
            tokenized_ids.append(ids)
            valid_samples_found += 1
        
        if valid_samples_found >= n_samples:
            print(f"Successfully collected {valid_samples_found} valid samples.")
            break
            
        if total_items_processed % 50000 == 0 and total_items_processed > 0:
            print(f"  ...processed {total_items_processed} items, found {valid_samples_found} valid samples...")



    print(f"Processing '{split}' complete. (Total sentences: {len(tokenized_ids)})")
    return tokenized_ids


tokenized_corpus_ids = load_and_tokenize("train", N_TRAIN_SAMPLES)
tokenized_validation_ids = load_and_tokenize("validation", N_VALID_SAMPLES)


   
print("\n--- Training Model ---")
model_4gram = NGramLanguageModel(n=4)
model_4gram.train(tokenized_corpus_ids)


print("\n--- General Validation PPL ---")
ppl_general = model_4gram.calculate_perplexity(tokenized_validation_ids, vocab_size)
print(f"\nOverall Perplexity (4-gram) on {N_VALID_SAMPLES} validation samples: {ppl_general:.2f}")



print("     Perplexity for Specific Sentences     ")



s1 = "آنها دوست داشتند در ماسه بازی کنند و جزر و مد آب را تماشا کنند"
s2 = "جیل و تام به همراه مامان و بابا به ساحل رفتند"
s3 = "تام بطری‌ نوشابه‌ را تا حد ممکن بالا انداخت و به سمت او فریاد زد"
s4 = "باری خیلی دوست داشت بیرون از منزل نقاشی کند و با پدربزرگ منظره تماشا کند"
sentences = [s1, s2, s3, s4]
sentence_names = ["S1 (جزر و مد)", "S2 (مامان و بابا)", "S3 (بطری نوشابه)", "S4 (پدربزرگ منظره)"]

results = {}

print("--- Calculating PPL with 4-gram model ---")

for name, sentence_text in zip(sentence_names, sentences):
    # Preprocess and tokenize the sentence
    cleaned_s = preprocess_pipeline(sentence_text)
    if not cleaned_s:
        results[name] = float('inf')
        continue
        
    token_ids = tokenizer.encode(cleaned_s).ids
    
  
    ppl = model_4gram.calculate_perplexity([token_ids], vocab_size)
    
    results[name] = ppl
    print(f"Perplexity {name}: {ppl:.2f}")

# Find the most probable sentence
if results:
    most_likely_name = min(results, key=results.get)
    most_likely_ppl = results[most_likely_name]

    print("\n--- Conclusion ---")
    print(f"**Most Likely Sentence (Lowest PPL): {most_likely_name}**")
    print(f"Perplexity: {most_likely_ppl:.2f}")


## 7. Four smoothing strategies compared

The same 4-gram counts scored four ways on the same validation set:

| Method | Perplexity |
|---|---|
| Unsmoothed MLE | ∞ |
| Laplace (add-1) | 3,494.58 |
| Interpolation | 38.31 |
| **Backoff** | **35.91** |

The gap between Laplace and the other two is the interesting part — roughly
**91×**, from the same counts. Laplace treats every unseen 4-gram as equally
likely. Interpolation and backoff instead consult the lower-order models, so an
unseen 4-gram whose trailing trigram is common is still rated as probable.
Backoff edges out interpolation by consulting shorter contexts only when the
longer one is missing, rather than always blending.

In [ ]:
import re
import os
import sys
import unicodedata
import random
import math
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from collections import defaultdict

class NGramCounter:

    def __init__(self, n):
        self.n = n
        self.model = defaultdict(lambda: defaultdict(int)) 
        self.context_totals = defaultdict(int)
        self.START_TOKEN = -1
        self.END_TOKEN = -2

    def _get_ngrams(self, token_ids_list):
        padding = [self.START_TOKEN] * (self.n - 1)
        full_sequence = padding + token_ids_list + [self.END_TOKEN]
        for i in range(len(full_sequence) - self.n + 1):
            ngram = tuple(full_sequence[i : i + self.n])
            yield ngram

    def train(self, tokenized_corpus):
        for token_ids_list in tokenized_corpus:
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                self.model[context][token] += 1
                self.context_totals[context] += 1


class SmoothedLanguageModel:
  
    def __init__(self, m1, m2, m3, m4, tokenizer):
        self.m1 = m1 # Unigram
        self.m2 = m2 # Bigram
        self.m3 = m3 # Trigram
        self.m4 = m4 # 4-gram
        self.models = {1: m1, 2: m2, 3: m3, 4: m4}
        
        self.tokenizer = tokenizer
        self.vocab_size = tokenizer.get_vocab_size()
        
        # Store total vocabulary (Unigram tokens)
        self.unigram_counts = m1.model[()]
        self.total_tokens = m1.context_totals[()]
        self.all_tokens_list = list(self.unigram_counts.keys()) # Required for generation
        
        self.START_TOKEN = m1.START_TOKEN
        self.END_TOKEN = m1.END_TOKEN
        
        # Model parameters
        self.interpolation_lambdas = {4: 0.4, 3: 0.3, 2: 0.2, 1: 0.1}
        self.backoff_lambda = 0.4

    def _get_contexts(self, context_ids):
        
        c4 = tuple(context_ids[-3:])
        c3 = tuple(context_ids[-2:])
        c2 = tuple(context_ids[-1:])
        c1 = ()
        return {4: c4, 3: c3, 2: c2, 1: c1}

    def _get_mle_prob(self, n, context, token):

        model = self.models[n]
        token_count = model.model[context].get(token, 0)
        
        if n == 1:
            # Laplace Smoothing for Unigram (P(w) > 0 is guaranteed)
            return (token_count + 1) / (self.total_tokens + self.vocab_size)
        
        # Standard MLE for N > 1
        context_total = model.context_totals.get(context, 0)
        
        if context_total == 0:
            return 0.0
        return token_count / context_total

    def get_probability(self, context_ids, token, method):

        contexts = self._get_contexts(context_ids)
        
        if method == "unsmoothed":
            return self._get_mle_prob(4, contexts[4], token)
            
        if method == "laplace":
            model = self.m4
            context = contexts[4]
            token_count = model.model[context].get(token, 0)
            context_total = model.context_totals.get(context, 0)
            return (token_count + 1) / (context_total + self.vocab_size)
            
        if method == "interpolation":
            total_prob = 0.0
            for n, lam in self.interpolation_lambdas.items():
                prob = self._get_mle_prob(n, contexts[n], token)
                total_prob += lam * prob
            return total_prob
            
        if method == "backoff":
            prob_n4 = self._get_mle_prob(4, contexts[4], token)
            if prob_n4 > 0:
                return prob_n4
            
            prob_n3 = self._get_mle_prob(3, contexts[3], token)
            if prob_n3 > 0:
                return self.backoff_lambda * prob_n3
                
            prob_n2 = self._get_mle_prob(2, contexts[2], token)
            if prob_n2 > 0:
                return self.backoff_lambda**2 * prob_n2
                
            # P(w) - Guaranteed P > 0
            prob_n1 = self._get_mle_prob(1, contexts[1], token)
            return self.backoff_lambda**3 * prob_n1
            
        raise ValueError(f"Unknown smoothing method: '{method}'")

    def _get_next_token(self, context_ids, method):
       
        tokens = []
        weights = []
        
        # Calculate probability for *every* possible token in the vocab
        
        for token_id in self.all_tokens_list:
            prob = self.get_probability(context_ids, token_id, method)
            tokens.append(token_id)
            weights.append(prob)
            
        total_weight = sum(weights)
        if total_weight == 0:
             
            
             return random.choice(self.all_tokens_list)
            
        # Weighted sampling
        next_token = random.choices(tokens, weights=weights, k=1)[0]
        return next_token

    def generate(self, method, num_tokens=100):
        
        print(f"\nGenerating text with {method}")
        # Start with the initial context (n-1 start tokens)
        context_ids = [self.START_TOKEN] * (self.m4.n - 1)
        generated_ids = []
        
        for i in range(num_tokens):
            
            
             next_token = self._get_next_token(context_ids, method)
            
             # If we hit the end token, reset the context
             if next_token == self.END_TOKEN:
                 context_ids = [self.START_TOKEN] * (self.m4.n - 1)
             else:
                 generated_ids.append(next_token)
                 context_ids = context_ids[1:] + [next_token]
        
        print("Text generation complete.")
        return generated_ids

    def calculate_perplexity(self, tokenized_validation_corpus, method):
        
        # print(f"Calculating Perplexity for {method}...")
        total_log_prob = 0.0
        total_token_count = 0
        n = self.m4.n # max_n = 4

        for token_ids_list in tokenized_validation_corpus:
            padding = [self.START_TOKEN] * (n - 1)
            full_sequence = padding + token_ids_list + [self.END_TOKEN]
            
            total_token_count += len(token_ids_list) + 1
            
            for i in range(n - 1, len(full_sequence)):
                context_ids = full_sequence[i - (n - 1) : i]
                token_id = full_sequence[i]
                
                probability = self.get_probability(context_ids, token_id, method)
                
                if probability == 0:
                    total_log_prob = -math.inf
                    break
                
                total_log_prob += math.log(probability)
            
            if total_log_prob == -math.inf:
                break
        
        if total_log_prob == -math.inf:
            return math.inf
            
        avg_neg_log_likelihood = -total_log_prob / total_token_count
        perplexity = math.exp(avg_neg_log_likelihood)
        
        return perplexity


def normalize_whitespace(text):
    if text is None: return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_pipeline(text):
    text = normalize_whitespace(text)
    return text




TOKENIZER_PATH = "TinyStories-Farsi-BPE.json"
N_TRAIN_SAMPLES = 50000 
N_VALID_SAMPLES = 5000 


if not os.path.exists(TOKENIZER_PATH):
    print(f"Error: File '{TOKENIZER_PATH}' not found.")
    sys.exit()

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = ByteLevelDecoder()
vocab_size = tokenizer.get_vocab_size()
print(f"BPE tokenizer loaded. (Vocabulary size: {vocab_size})")


def load_and_tokenize(split, n_samples):
    tokenized_ids = []
 
    dataset = load_dataset("taesiri/TinyStories-Farsi", split=split, streaming=True)
    for example in dataset.take(n_samples):
        cleaned_text = preprocess_pipeline(example['Persian'])
        if cleaned_text:
            ids = tokenizer.encode(cleaned_text).ids
            tokenized_ids.append(ids)
    return tokenized_ids
    

tokenized_corpus_ids = load_and_tokenize("train", N_TRAIN_SAMPLES)
tokenized_validation_ids = load_and_tokenize("validation", N_VALID_SAMPLES)



model_1gram = NGramCounter(n=1)
model_1gram.train(tokenized_corpus_ids)

model_2gram = NGramCounter(n=2)
model_2gram.train(tokenized_corpus_ids)

model_3gram = NGramCounter(n=3)
model_3gram.train(tokenized_corpus_ids)

model_4gram = NGramCounter(n=4)
model_4gram.train(tokenized_corpus_ids)


smoother = SmoothedLanguageModel(model_1gram, model_2gram, model_3gram, model_4gram, tokenizer)



print("     Starting text generation (this is slow)     ")



text_laplace_ids = smoother.generate('laplace', 100)
text_interpolation_ids = smoother.generate('interpolation', 100)
text_backoff_ids = smoother.generate('backoff', 100)
text_unsmoothed_ids = smoother.generate('unsmoothed', 100) 

text_laplace = tokenizer.decode(text_laplace_ids)
text_interpolation = tokenizer.decode(text_interpolation_ids)
text_backoff = tokenizer.decode(text_backoff_ids)
text_unsmoothed = tokenizer.decode(text_unsmoothed_ids)


print("           Output 1: Generated Texts (N=4)          ")


print(" Unsmoothed Model ")
print(text_unsmoothed)

print("\n--- Laplace (Add-1) Model ---")
print(text_laplace)

print("Interpolation Model")
print(text_interpolation)

print(" Backoff Model ")
print(text_backoff)



print("     Starting Perplexity calculation on Validation data     ")


ppl_unsmoothed = smoother.calculate_perplexity(tokenized_validation_ids, 'unsmoothed')
ppl_laplace = smoother.calculate_perplexity(tokenized_validation_ids, 'laplace')
ppl_interpolation = smoother.calculate_perplexity(tokenized_validation_ids, 'interpolation')
ppl_backoff = smoother.calculate_perplexity(tokenized_validation_ids, 'backoff')


print("          Output 2: Perplexity Results (N=4)          ")


print(f"Perplexity (Unsmoothed):   {ppl_unsmoothed:.2f}")
print(f"Perplexity (Laplace):      {ppl_laplace:.2f}")
print(f"Perplexity (Interpolation):{ppl_interpolation:.2f}")
print(f"Perplexity (Backoff):      {ppl_backoff:.2f}")

### Temperature-controlled generation

Dividing the logits by a temperature before sampling. Below 1.0 sharpens the
distribution toward the most likely token (repetitive but fluent); above 1.0
flattens it (varied but less coherent).

In [ ]:
import re
import os
import sys
import unicodedata
import random
import math
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from collections import defaultdict


class NGramCounter:
    
    def __init__(self, n):
        self.n = n
        self.model = defaultdict(lambda: defaultdict(int)) 
        self.context_totals = defaultdict(int) 
        self.START_TOKEN = -1
        self.END_TOKEN = -2

    def _get_ngrams(self, token_ids_list):
        padding = [self.START_TOKEN] * (self.n - 1)
        full_sequence = padding + token_ids_list + [self.END_TOKEN]
        for i in range(len(full_sequence) - self.n + 1):
            ngram = tuple(full_sequence[i : i + self.n])
            yield ngram

    def train(self, tokenized_corpus):
        print(f"Starting {self.n}-gram counting...")
        for token_ids_list in tokenized_corpus:
            for ngram in self._get_ngrams(token_ids_list):
                context = ngram[:-1]
                token = ngram[-1]
                self.model[context][token] += 1
                self.context_totals[context] += 1
        print(f"{self.n}-gram counting complete. Contexts found: {len(self.model)}")



class SmoothedLanguageModel:
   
    def __init__(self, m1, m2, m3, m4, tokenizer):
        print("Building smoothed language model...")
        self.m1 = m1 # Unigram
        self.m2 = m2 # Bigram
        self.m3 = m3 # Trigram
        self.m4 = m4 # 4-gram
        self.models = {1: m1, 2: m2, 3: m3, 4: m4}
        
        self.tokenizer = tokenizer
        self.vocab_size = tokenizer.get_vocab_size()
        
        self.unigram_counts = m1.model[()]
        self.total_tokens = m1.context_totals[()]
        self.all_tokens_list = list(self.unigram_counts.keys())
        
        self.START_TOKEN = m1.START_TOKEN
        self.END_TOKEN = m1.END_TOKEN
        
        # Model parameters
        self.interpolation_lambdas = {4: 0.4, 3: 0.3, 2: 0.2, 1: 0.1}
        self.backoff_lambda = 0.4

    def _get_contexts(self, context_ids):
       
        c4 = tuple(context_ids[-3:])
        c3 = tuple(context_ids[-2:])
        c2 = tuple(context_ids[-1:])
        c1 = ()
        return {4: c4, 3: c3, 2: c2, 1: c1}

    def _get_mle_prob(self, n, context, token):
      
        
       # P(token | context) = count(context, token) / count(context)
       
        model = self.models[n]
        token_count = model.model[context].get(token, 0)
        context_total = model.context_totals.get(context, 0)
        
        if context_total == 0:
            return 0.0
        return token_count / context_total

    def get_probability(self, context_ids, token, method):
     
        contexts = self._get_contexts(context_ids)
        
        if method == "unsmoothed":
            return self._get_mle_prob(4, contexts[4], token)
            
        if method == "laplace":
            model = self.m4
            context = contexts[4]
            token_count = model.model[context].get(token, 0)
            context_total = model.context_totals.get(context, 0)
            return (token_count + 1) / (context_total + self.vocab_size)
            
        if method == "interpolation":
            total_prob = 0.0
            for n, lam in self.interpolation_lambdas.items():
                prob = self._get_mle_prob(n, contexts[n], token)
                total_prob += lam * prob
            return total_prob
            
        if method == "backoff":
            prob_n4 = self._get_mle_prob(4, contexts[4], token)
            if prob_n4 > 0: return prob_n4
            prob_n3 = self._get_mle_prob(3, contexts[3], token)
            if prob_n3 > 0: return self.backoff_lambda * prob_n3
            prob_n2 = self._get_mle_prob(2, contexts[2], token)
            if prob_n2 > 0: return self.backoff_lambda * self.backoff_lambda * prob_n2
            prob_n1 = self._get_mle_prob(1, contexts[1], token)
            return self.backoff_lambda * self.backoff_lambda * self.backoff_lambda * prob_n1
            
        raise ValueError(f"Unknown smoothing method: '{method}'")


    def _get_next_token_with_temp(self, context_ids, method, temperature):
    
        if temperature <= 0:
            temperature = 0.01 # Prevent division by zero
            
        tokens = []
        logits = [] # Scores (Log-Probabilities)

        #  Get probabilities and convert to logits
        for token_id in self.all_tokens_list:
            prob = self.get_probability(context_ids, token_id, method)
            if prob > 0: # Avoid log(0)
                tokens.append(token_id)
                logits.append(math.log(prob)) # l_i = log(P_i)
        
        if not logits:
            return random.choice(self.all_tokens_list)

   
        scaled_logits = [l / temperature for l in logits]

    
        max_logit = max(scaled_logits)
        exp_logits = [math.exp(l - max_logit) for l in scaled_logits]
        sum_exp_logits = sum(exp_logits)
        
        if sum_exp_logits == 0:
             return random.choice(self.all_tokens_list)

    
        new_probabilities = [e / sum_exp_logits for e in exp_logits]

        #  Final sampling
        next_token = random.choices(tokens, weights=new_probabilities, k=1)[0]
        return next_token

  
    def generate(self, method, num_tokens=100, temperature=1.0):
     
        print(f"\n---  Generating text with {method} (T={temperature}) ---")
        context_ids = [self.START_TOKEN] * (self.m4.n - 1)
        generated_ids = []
        
        for i in range(num_tokens):
            if num_tokens > 50 and i % 10 == 0: 
                print(f"  ... generating token {i}/{num_tokens}")

            # Call the new method
            next_token = self._get_next_token_with_temp(context_ids, method, temperature)
            
            if next_token == self.END_TOKEN:
                context_ids = [self.START_TOKEN] * (self.m4.n - 1)
            else:
                generated_ids.append(next_token)
                context_ids = context_ids[1:] + [next_token]
        
        print(f"Text generation (T={temperature}) complete.")
        return generated_ids


def normalize_unicode(text):
    if text is None: return ""
    return unicodedata.normalize('NFC', text)

def remove_kashida(text):
    if text is None: return ""
    return text.replace('\u0640', '')

def normalize_whitespace(text):
    if text is None: return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_pipeline(text):
    text = normalize_unicode(text)
    text = remove_kashida(text)
    text = normalize_whitespace(text)
    return text


TOKENIZER_PATH = "TinyStories-Farsi-BPE.json"
N_TRAIN_SAMPLES = 50000 # Number of sentences for training


tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = ByteLevelDecoder()
vocab_size = tokenizer.get_vocab_size()
print(f"BPE tokenizer loaded. (Vocabulary size: {vocab_size})")


def load_and_tokenize(split, n_samples):
    print(f"Loading and processing {n_samples} samples from '{split}' split...")
    tokenized_ids = []

    dataset = load_dataset("taesiri/TinyStories-Farsi", split=split, streaming=True)
    for example in dataset.take(n_samples):
        cleaned_text = preprocess_pipeline(example['Persian'])
        if cleaned_text:
            ids = tokenizer.encode(cleaned_text).ids
            tokenized_ids.append(ids)
    print(f"Processing '{split}' complete. (Sentences: {len(tokenized_ids)})")
    return tokenized_ids
    


tokenized_corpus_ids = load_and_tokenize("train", N_TRAIN_SAMPLES)
if not tokenized_corpus_ids:
    print("Error: Training data not loaded. Exiting.")
    sys.exit()


model_1gram = NGramCounter(n=1)
model_1gram.train(tokenized_corpus_ids)

model_2gram = NGramCounter(n=2)
model_2gram.train(tokenized_corpus_ids)

model_3gram = NGramCounter(n=3)
model_3gram.train(tokenized_corpus_ids)

model_4gram = NGramCounter(n=4)
model_4gram.train(tokenized_corpus_ids)

smoother = SmoothedLanguageModel(model_1gram, model_2gram, model_3gram, model_4gram, tokenizer)


LOW_TEMP = 0.01    
HIGH_TEMP = 3.0    
NUM_TOKENS = 20    
NUM_RUNS = 3       

texts_low_temp = []
texts_high_temp = []


print(f"     Generating text with Low Temperature (T={LOW_TEMP})     ")

for i in range(NUM_RUNS):
    ids = smoother.generate('interpolation', NUM_TOKENS, LOW_TEMP)
    texts_low_temp.append(tokenizer.decode(ids))
    

print(f"     Generating text with High Temperature (T={HIGH_TEMP})     ")

for i in range(NUM_RUNS):
    ids = smoother.generate('interpolation', NUM_TOKENS, HIGH_TEMP)
    texts_high_temp.append(tokenizer.decode(ids))



print("          Expected Output: Generated Texts        ")


print(f"\n---  Generated Texts with Low Temperature (T={LOW_TEMP}) ---")
for i, text in enumerate(texts_low_temp):
    print(f"--- Text {i+1} ---")
    print(text)
    print("-" * 20)

print(f"\n---  Generated Texts with High Temperature (T={HIGH_TEMP}) ---")
for i, text in enumerate(texts_high_temp):
    print(f"--- Text {i+1} ---")
    print(text)
    print("-" * 20)

